# 📓 Semana 15 · Dia 5 — Segurança empresarial: sandbox, PII e guardrails avançados

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition (conceito) + 🔑 (avançado) |
| **Tempo estimado** | 2h |
| **Certificação alvo** | GenAI Engineer Associate |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Política de segurança do agente |

---


## 📖 Teoria — As camadas de segurança de um agente

1. **Entrada**: guardrails (tópicos, prompt injection)
2. **Ferramentas**: permissões UC + SELECT only
3. **Execução**: **sandbox Python isolado** — código gerado roda sem acessar o resto do workspace
4. **Saída**: PII masking, filtro de conteúdo
5. **Auditoria**: tudo logado


### 💻 Na prática — Sandbox e permissões

Código do agente (UDF/ferramentas) roda isolado — conceito e prática na Free.


In [ ]:
# Ferramentas com escopo mínimo (princípio do menor privilégio)
# A tool SÓ lê a tabela Ouro — nunca acessa o workspace inteiro
def tool_segura(pais: str) -> str:
    """Consulta receita de um país (somente leitura)."""
    return spark.sql(f"SELECT receita_total FROM workspace.ouro.receita_por_pais WHERE UPPER(Country) = UPPER('{pais}')").collect()[0][0]
print("Tool com escopo mínimo: só leitura do Ouro.")

In [ ]:
# Guardrails avançados (entrada + saída)
def guardrail_avancado(pergunta, resposta):
    # Entrada: bloqueia prompt injection
    sinais = ["ignore previous", "ignore as instruções", "reveal your prompt"]
    if any(s in pergunta.lower() for s in sinais):
        return False, "Bloqueado: possível prompt injection."
    # Saída: bloqueia PII
    if detecta_pii(resposta)["email"] or detecta_pii(resposta)["cpf"]:
        return False, "Bloqueado: resposta com PII."
    return True, resposta
print("Guardrail duplo (entrada + saída) implementado.")

### 💻 Na prática — Auditoria total

Registre cada decisão do guardrail, cada tool chamada e cada resposta.


In [ ]:
# Auditoria enriquecida
def registrar_auditoria(pergunta, resposta, tools_usadas, ok, motivo=""):
    spark.createDataFrame([(
        "now", pergunta, resposta, str(tools_usadas), ok, motivo
    )], ["ts", "pergunta", "resposta", "tools", "ok", "motivo"])\
        .withColumn("ts", current_timestamp())\
        .write.mode("append").saveAsTable("workspace.audit.log_agente_completo")
print("Auditoria completa: workspace.audit.log_agente_completo")

> 🎯 **Dica de prova**: Segurança de agentes: sandbox (execução isolada), PII (saída), guardrails (entrada/saída), menor privilégio (tools). Pergunta: 'como isolar a execução de código do agente?' → sandbox.


## 🎯 Exercícios de fixação

**1.** O que o sandbox Python isola?

**2.** Por que o guardrail de saída é obrigatório?

**3.** Monte a política de segurança do agente (documento).


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Sandbox

Executa código (UDF/tools) num ambiente isolado sem acesso ao workspace/credenciais — evita exfiltração.

**2.** Saída obrigatória

O LLM pode vazar PII vinda do contexto — filtrar a resposta é a última linha antes do usuário.

**3.** Política

Documente: quem pode usar, quais tools, o que é bloqueado, retenção de logs, processo de incidente.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*